# MoveScope Viewpoint Robustness

Checks whether 3D full assessment is more consistent than the 2D baseline across front, side, and diagonal squat videos.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "movescope").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from movescope.experiments import run_ablation, summarize_ablation, run_viewpoint_consistency, viewpoint_std
from movescope.template import ActionTemplate

FIG_DIR = ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
template_path = Path("data/templates/squat.npz")
if not template_path.exists():
    print("Missing data/templates/squat.npz. Build it with scripts/build_template.py before running this experiment.")
    rows = []
else:
    template = ActionTemplate.load("squat")
    rows = run_viewpoint_consistency(template, multiview_dir="data/test/multiview")
    print(f"Collected {len(rows)} scored rows")


In [ ]:
df = pd.DataFrame(rows)
if df.empty:
    print("No multiview videos found. Add front/side/diagonal videos to data/test/multiview first.")
else:
    display(df)
    print(viewpoint_std(rows))


In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    for variant, group in df.groupby("variant"):
        group = group.sort_values("angle")
        ax.plot(group["angle"], group["total_score"], marker="o", label=variant)
    ax.set_title("Score consistency across camera viewpoints")
    ax.set_xlabel("viewpoint")
    ax.set_ylabel("total_score")
    ax.legend()
    fig.tight_layout()
    output = FIG_DIR / "viewpoint_robustness.png"
    fig.savefig(output, dpi=160)
    print(f"Saved {output}")


In [ ]:
if not df.empty:
    stds = viewpoint_std(rows)
    baseline = stds.get("baseline_2d")
    full = stds.get("ours_full")
    if baseline and full is not None:
        improvement = (baseline - full) / baseline * 100.0
        print(f"ours_full consistency improvement: {improvement:.1f}%")
